## WAP to implement linear regression.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

In [2]:
spark = SparkSession.builder.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/04/03 21:47:13 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
data = spark.read.csv('datasets/Walmart_sales.csv', header=True, inferSchema=True)

In [4]:
data.printSchema()
data.show(5)

root
 |-- Store: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Weekly_Sales: double (nullable = true)
 |-- Holiday_Flag: integer (nullable = true)
 |-- Temperature: double (nullable = true)
 |-- Fuel_Price: double (nullable = true)
 |-- CPI: double (nullable = true)
 |-- Unemployment: double (nullable = true)

+-----+----------+------------+------------+-----------+----------+-----------+------------+
|Store|      Date|Weekly_Sales|Holiday_Flag|Temperature|Fuel_Price|        CPI|Unemployment|
+-----+----------+------------+------------+-----------+----------+-----------+------------+
|    1|05-02-2010|   1643690.9|           0|      42.31|     2.572|211.0963582|       8.106|
|    1|12-02-2010|  1641957.44|           1|      38.51|     2.548|211.2421698|       8.106|
|    1|19-02-2010|  1611968.17|           0|      39.93|     2.514|211.2891429|       8.106|
|    1|26-02-2010|  1409727.59|           0|      46.63|     2.561|211.3196429|       8.106|
|    1|05-03-201

In [5]:
features = data.columns
features.remove('Date')
features.remove('Weekly_Sales')

In [6]:
assembler = VectorAssembler(inputCols=features, outputCol='features')
evaluator = RegressionEvaluator(predictionCol='prediction', labelCol='Weekly_Sales', metricName='rmse')

In [7]:
data = assembler.transform(data)
train_data, test_data = data.randomSplit([0.8, 0.2], seed=100)

In [12]:
lr = LinearRegression(featuresCol='features', labelCol='Weekly_Sales', predictionCol = 'prediction', maxIter=50, regParam=0.01)
lr_model = lr.fit(train_data)
lr_pred = lr_model.transform(test_data)

In [13]:
evaluator.evaluate(lr_pred)

521441.7933884186